# MiniLM Cross-Encoder Sentence Reranker — final POST-W12 baseline

**POST_W12_RERANKER_BASELINE_EXPLORATORY**

This is a POST-W12 exploratory matched-cohort comparison, not part of frozen Week-12 method selection or confirmation. It uses the same previously analyzed 50-query-per-dataset cohort as post-hoc BEA_V2. Existing historical answers, splits and the official FIXED_SENTENCE final policy remain unchanged. BEA_V2's preregistered calibration remains CALIBRATION_FAILED_TERMINATED.

**To run:** set only `RUN_FULL=True`, then **Run All**. All 150 MiniLM score caches and 300 answer rows resume automatically. The final comparison appears in the last section. Opening/running the default notebook performs only readiness inspection; it never launches the full experiment.


In [1]:
from pathlib import Path
import sys, os, json, importlib.metadata
sys.dont_write_bytecode = True
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
from IPython.display import display, Markdown
import pandas as pd
ROOT = Path.cwd().resolve()
while not (ROOT/'W12-POST/reranker_baseline').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open inside CS787-RAG-Project')
    ROOT = ROOT.parent
for path in [ROOT,ROOT/'W12-POST']:
    if str(path) not in sys.path:
        sys.path.insert(0,str(path))
RUN_PREFLIGHT = True
RUN_SMOKE = False
RUN_READER_SMOKE = False
RUN_FULL = True  # Change ONLY this to True, then Run All.
RUN_ANALYSIS = RUN_FULL
ALLOW_MODEL_DOWNLOAD = True
RUN_INSTALL_DEPENDENCIES = True
RESUME = True
OVERWRITE = False
if OVERWRITE or not RESUME:
    raise ValueError('Immutable resume only; do not overwrite existing results')
if RUN_READER_SMOKE and not RUN_SMOKE:
    raise ValueError('Reader smoke requires RUN_SMOKE=True; at most three answers @256')
from reranker_baseline.protocol import OUT, LABEL, MODEL, REVISION, NOTICE
print(LABEL, '\n', NOTICE)
print('Model:', MODEL, '@', REVISION)
print('Outputs:', OUT)


POST_W12_RERANKER_BASELINE_EXPLORATORY 
 POST-W12 exploratory matched-cohort comparison on previously analyzed development data. No fresh holdout, confirmation or test claim. Official Week-12 policy unchanged.
Model: cross-encoder/ms-marco-MiniLM-L6-v2 @ 233902d25c440f23af6f7d6e94d2946bac0bee0a
Outputs: /home/gunavenkat/Downloads/CS787-RAG-Project/W12-POST/results/minilm_reranker


## B. Environment and optional isolated dependency install

Installed dependencies are inspected below. No model is loaded in this cell. The cross-encoder runs in CPU FP32; canonical Qwen requires GPU only when explicitly generating reader answers. If a dependency is missing, the optional installation uses a local target under W12-POST and is disabled by default; review compatibility with the frozen environment before using it. No global package upgrade is performed.


In [2]:
if RUN_INSTALL_DEPENDENCIES:
    import subprocess
    target = ROOT/'W12-POST/vendor'
    subprocess.run([str(ROOT/'sara_env/bin/python'),'-m','pip','install','--no-deps','--target',str(target),'sentence-transformers==5.7.0'],check=True)
    print('Installed only into:', target, '— review and restart the kernel before adjusting its import path.')
for name in ['torch','transformers','sentence-transformers','huggingface_hub']:
    try:
        print(name, importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(f'{name} missing; use an isolated reviewed install, not a global upgrade')
import torch
print('CUDA available for the later reader:', torch.cuda.is_available())
print('Reranker device: cpu; dtype: float32. Model weights are lazy-loaded only by explicit scoring.')
print('MiniLM local cache present:', (OUT/'model_cache').exists())


  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
Using cached sentence_transformers-5.7.0-py3-none-any.whl (611 kB)
Installed only into: /home/gunavenkat/Downloads/CS787-RAG-Project/W12-POST/vendor — review and restart the kernel before adjusting its import path.
torch 2.14.0
transformers 4.57.6
sentence-transformers 5.7.0
huggingface_hub 0.36.2



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


CUDA available for the later reader: True
Reranker device: cpu; dtype: float32. Model weights are lazy-loaded only by explicit scoring.
MiniLM local cache present: True


## C/D. Exact cohort and frozen SFR top-10 sources

The preflight asserts matching query/budget/group membership across all four saved controls; exactly 50 queries/dataset; identical frozen SFR encoder/candidate identities; exact source sentence boundaries; and unchanged reader/source hashes. Missing old caches cause an error, not new retrieval. No Qwen or CE inference occurs here.


In [3]:
from reranker_baseline.runner import preflight, smoke, prepare_full, generate_full
plan = None
if RUN_PREFLIGHT or RUN_FULL:
    plan = preflight()
    display(pd.DataFrame(plan['cohorts']))
    display({k:v for k,v in plan.items() if k!='cohorts'})


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,dataset,n_queries,n_groups,budgets
0,HotpotQA,50,50,"[256, 1024]"
1,QASPER,50,50,"[256, 1024]"
2,SciFact,50,50,"[256, 1024]"


{'label': 'POST_W12_RERANKER_BASELINE_EXPLORATORY',
 'notice': 'POST-W12 exploratory matched-cohort comparison on previously analyzed development data. No fresh holdout, confirmation or test claim. Official Week-12 policy unchanged.',
 'identity_hash': '2923313d7e14eb6d71f36a57a3f52d17bb25bdba371ecbdb9fdb6946152ddb25',
 'matched_control_rows': 1200,
 'expected_minilm_rows': 300,
 'reranker_queries_scored': 3,
 'configured_retrieval_width': 10,
 'short_frozen_pools': [{'dataset': 'QASPER',
   'query_id': '78292bc57ee68fdb93ed45430d80acca25a9e916',
   'available_passages': 8},
  {'dataset': 'QASPER',
   'query_id': 'f20a389ace2267aa61eddcc235535452ccdae0e6',
   'available_passages': 7}],
 'completed_answer_rows': 0,
 'missing_answer_upper_bound': 300,
 'free_disk_gib': 32.00178527832031,
 'model': 'cross-encoder/ms-marco-MiniLM-L6-v2',
 'revision': '233902d25c440f23af6f7d6e94d2946bac0bee0a',
 'dependencies': {'torch': '2.14.0',
  'transformers': '4.57.6',
  'sentence-transformers': '5.7.

## E/F. Bounded reranking + exact-budget smoke inspection

Optional smoke is **three total queries**, one/dataset, on frozen candidates. Scores are reused for both 256/1024 allocations. It records every score, tie order, original pair length/truncation, include/skip decision and exact evidence context. Gold/answer metrics/LOO/P_help do not enter scoring or packing. This default smoke does not generate reader answers.


In [4]:
smoke_result = None
if RUN_SMOKE:
    smoke_result = smoke(allow_download=ALLOW_MODEL_DOWNLOAD, reader=RUN_READER_SMOKE)
    for ex in smoke_result['examples']:
        display(Markdown('### '+ex['dataset']+' · '+ex['query_id']))
        print(ex['question'])
        print('Loaded model:', ex['model_info'])
        display(pd.DataFrame(ex['top10'])[['sentence_id','passage_rank','source_position','cross_encoder_score','sentence_text']])
        for budget in ['256','1024']:
            allocation = ex['allocation'][budget]
            print('Budget', budget, '| evidence tokens', allocation['evidence_tokens'])
            display(pd.DataFrame(allocation['decisions']))
            print('Exact source-order evidence:\n', allocation['text'])
else:
    print('Smoke disabled. The implementation smoke report, if present, is available under smoke/SMOKE_REPORT.json.')


Smoke disabled. The implementation smoke report, if present, is available under smoke/SMOKE_REPORT.json.


## G. Optional reader smoke — disabled

Set both `RUN_SMOKE=True` and `RUN_READER_SMOKE=True` above to manually generate at most 3 Qwen answers @256. This is only a plumbing check, never scientific evidence from three queries. The implementation handoff smoke is cross-encoder/scoring/packing only and does not load Qwen.


In [5]:
if smoke_result and RUN_READER_SMOKE:
    for ex in smoke_result['examples']:
        row = ex['reader_result']
        display(dict(dataset=ex['dataset'], query=ex['question'], gold=row['gold'], evidence=row['evidence_text'],
                     evidence_tokens=row['evidence_tokens'], answer=row['generated_answer'],
                     score=row['correct'] if ex['dataset']=='SciFact' else row['f1']))
else:
    print('No reader smoke requested.')


No reader smoke requested.


## H. Explicit full run — 300 answer conditions

CPU MiniLM scoring and both-budget fit/skip packing are completed first; MiniLM is released before the frozen GPU reader loads. Exact completed rows and compatible prompt caches resume. Existing four-method answers are read-only and never regenerated. The full run needs a free GPU with ≥18 GiB headroom for Qwen BASE_FP16. No automatic precision/model substitution or continuation to a new experiment is allowed.


In [6]:
if RUN_FULL:
    prepared = prepare_full(explicit=True, allow_download=ALLOW_MODEL_DOWNLOAD)
    print('Prepared/reused selection conditions:', len(prepared))
    display(generate_full(explicit=True))
else:
    print('Full run disabled. Set RUN_FULL=True above, then Run All when ready.')


MiniLM full: prepared/reused 1/150 queries; scores shared by both budgets


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1003.50it/s]
The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.


Cross-encoder: {'label': 'POST_W12_RERANKER_BASELINE_EXPLORATORY', 'model': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'revision': '233902d25c440f23af6f7d6e94d2946bac0bee0a', 'device': 'cpu', 'dtype': 'torch.float32', 'parameter_count': 22713601, 'max_length': 512, 'activation': 'raw_logit_identity', 'training': False}
MiniLM full: prepared/reused 2/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 3/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 4/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 5/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 6/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 7/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 8/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 9/150 queries; scores shared by both budgets
MiniLM full: prepared/reused 10/150 queries; scores shared by both budgets
MiniLM full: pre

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.90it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MiniLM full durable answers 1/300; elapsed=5s
MiniLM full durable answers 2/300; elapsed=6s
MiniLM full durable answers 3/300; elapsed=7s
MiniLM full durable answers 4/300; elapsed=8s
MiniLM full durable answers 5/300; elapsed=9s
MiniLM full durable answers 6/300; elapsed=9s
MiniLM full durable answers 7/300; elapsed=9s
MiniLM full durable answers 8/300; elapsed=9s
MiniLM full durable answers 9/300; elapsed=10s
MiniLM full durable answers 10/300; elapsed=10s
MiniLM full durable answers 11/300; elapsed=10s
MiniLM full durable answers 12/300; elapsed=10s
MiniLM full durable answers 13/300; elapsed=11s
MiniLM full durable answers 14/300; elapsed=11s
MiniLM full durable answers 15/300; elapsed=11s
MiniLM full durable answers 16/300; elapsed=11s
MiniLM full durable answers 17/300; elapsed=12s
MiniLM full durable answers 18/300; elapsed=13s
MiniLM full durable answers 19/300; elapsed=13s
MiniLM full durable answers 20/300; elapsed=14s
MiniLM full durable answers 21/300; elapsed=14s
MiniLM fu

{'label': 'POST_W12_RERANKER_BASELINE_EXPLORATORY',
 'completed': 300,
 'mode': 'full'}

## I. Final five-method comparison, paired uncertainty and closeout

The table below is produced only after the exact 300-row reranker surface is complete. HotpotQA/QASPER use answer F1; SciFact uses aggregate three-class macro-F1. Highest point estimates are bold. Win counts are descriptive only, with all exact ties credited. Paired group-aware inference uses 10,000 resamples, seed 42; one exploratory Holm family contains all 24 MiniLM-versus-control contrasts. No cross-dataset mean is calculated.


In [7]:
RUN_ANALYSIS=True
if RUN_ANALYSIS:
    from reranker_baseline.analysis import analyze
    result = analyze()
    display(Markdown(result['final_table'].to_markdown(index=False)))
    display(result['efficiency'])
    display(pd.DataFrame([dict(Method=m, Wins=f'{n} / 6') for m,n in result['wins'].items()]))
    print('Descriptive point-estimate wins only; ties credit all tied methods.')
    display(result['summary'])
    display(result['contrasts'])
    display(Markdown(result['report'].read_text()))
else:
    print('Final comparison pending full completion. To analyze completed caches only, set RUN_ANALYSIS=True.')


| Dataset   |   Budget | FIXED      | QBGF       |   BEA-V1 | BEA-V2     | MiniLM Reranker   | Winner          |
|:----------|---------:|:-----------|:-----------|---------:|:-----------|:------------------|:----------------|
| HotpotQA  |      256 | 0.3661     | 0.4042     |   0.2976 | **0.4352** | 0.3404            | BEA-V2          |
| HotpotQA  |     1024 | 0.3893     | 0.4020     |   0.3441 | **0.4234** | 0.3618            | BEA-V2          |
| QASPER    |      256 | **0.2668** | 0.2641     |   0.1868 | 0.2618     | 0.2641            | FIXED           |
| QASPER    |     1024 | 0.2896     | **0.2964** |   0.2084 | 0.2904     | 0.2827            | QBGF            |
| SciFact   |      256 | 0.6289     | 0.5310     |   0.6127 | 0.5522     | **0.6855**        | MiniLM Reranker |
| SciFact   |     1024 | 0.5533     | 0.5881     |   0.6432 | 0.5772     | **0.6479**        | MiniLM Reranker |

,label,Dataset,Budget,Method,Quality,Mean Evidence Tokens,Budget Utilization,Selected Units,Represented Parents,mean_redundancy
0,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,BEA_QBGF_V1,0.297642,225.26,0.879922,18.16,9.66,0.594611
1,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,BEA_V2,0.435211,246.88,0.964375,7.46,4.98,0.611181
2,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,FIXED_SENTENCE,0.366109,254.84,0.995469,9.80,5.28,0.650451
3,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,0.340368,254.86,0.995547,10.54,6.46,0.624712
4,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,QBGF_FULL,0.404225,251.62,0.982891,11.74,4.94,0.627003
5,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,BEA_QBGF_V1,0.344117,284.64,0.277969,20.02,9.72,0.591976
6,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,BEA_V2,0.423387,960.66,0.938145,16.20,9.50,0.572894
7,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,FIXED_SENTENCE,0.389292,1001.74,0.978262,36.06,9.62,0.585496
8,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,0.361826,1000.30,0.976855,38.16,9.94,0.574948
9,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,QBGF_FULL,0.402047,995.26,0.971934,38.28,9.24,0.573011


,Method,Wins
0,FIXED_SENTENCE,1 / 6
1,QBGF_FULL,1 / 6
2,BEA_QBGF_V1,0 / 6
3,BEA_V2,2 / 6
4,MINILM_CE_SENTENCE,2 / 6


Descriptive point-estimate wins only; ties credit all tied methods.


,label,dataset,budget,method,n_queries,n_groups,metric,quality,EM,accuracy,invalid_format,mean_prompt_tokens,mean_generated_tokens,conceptual_reader_calls
0,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,BEA_QBGF_V1,50,50,f1,0.297642,0.18,NaN,NaN,280.14,16.92,50
1,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,BEA_V2,50,50,f1,0.435211,0.30,NaN,NaN,301.76,12.44,50
2,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,FIXED_SENTENCE,50,50,f1,0.366109,0.22,NaN,NaN,309.72,13.54,50
3,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,50,50,f1,0.340368,0.18,NaN,NaN,309.74,15.08,50
4,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,QBGF_FULL,50,50,f1,0.404225,0.28,NaN,NaN,306.50,13.60,50
5,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,BEA_QBGF_V1,50,50,f1,0.344117,0.22,NaN,NaN,339.52,15.76,50
6,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,BEA_V2,50,50,f1,0.423387,0.28,NaN,NaN,1015.54,14.60,50
7,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,FIXED_SENTENCE,50,50,f1,0.389292,0.24,NaN,NaN,1056.62,15.34,50
8,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,50,50,f1,0.361826,0.22,NaN,NaN,1055.18,15.00,50
9,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,QBGF_FULL,50,50,f1,0.402047,0.26,NaN,NaN,1050.14,15.28,50


,label,dataset,budget,left,right,point,ci_low,ci_high,raw_p,resamples,seed,n_queries,n_clusters,metric,test,holm_p,exploratory_family_size
0,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,FIXED_SENTENCE,-0.025741,-0.107376,0.053458,0.558844,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
1,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,QBGF_FULL,-0.063857,-0.171352,0.039398,0.247275,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
2,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,BEA_QBGF_V1,0.042726,-0.041606,0.133215,0.346065,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
3,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,256,MINILM_CE_SENTENCE,BEA_V2,-0.094843,-0.217733,0.025721,0.143486,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
4,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,FIXED_SENTENCE,-0.027466,-0.097594,0.032973,0.476652,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
5,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,QBGF_FULL,-0.040222,-0.115768,0.026560,0.289971,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
6,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,BEA_QBGF_V1,0.017708,-0.064906,0.103461,0.689131,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
7,POST_W12_RERANKER_BASELINE_EXPLORATORY,HotpotQA,1024,MINILM_CE_SENTENCE,BEA_V2,-0.061562,-0.147642,0.014689,0.144186,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
8,POST_W12_RERANKER_BASELINE_EXPLORATORY,QASPER,256,MINILM_CE_SENTENCE,FIXED_SENTENCE,-0.002684,-0.025452,0.018551,0.819818,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24
9,POST_W12_RERANKER_BASELINE_EXPLORATORY,QASPER,256,MINILM_CE_SENTENCE,QBGF_FULL,-0.000009,-0.045150,0.039819,0.999400,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,1.000000,24


# MiniLM cross-encoder: matched-cohort comparison

POST-W12 exploratory matched-cohort comparison on previously analyzed development data. No fresh holdout, confirmation or test claim. Official Week-12 policy unchanged.

| Dataset   |   Budget | FIXED      | QBGF       |   BEA-V1 | BEA-V2     | MiniLM Reranker   | Winner          |
|:----------|---------:|:-----------|:-----------|---------:|:-----------|:------------------|:----------------|
| HotpotQA  |      256 | 0.3661     | 0.4042     |   0.2976 | **0.4352** | 0.3404            | BEA-V2          |
| HotpotQA  |     1024 | 0.3893     | 0.4020     |   0.3441 | **0.4234** | 0.3618            | BEA-V2          |
| QASPER    |      256 | **0.2668** | 0.2641     |   0.1868 | 0.2618     | 0.2641            | FIXED           |
| QASPER    |     1024 | 0.2896     | **0.2964** |   0.2084 | 0.2904     | 0.2827            | QBGF            |
| SciFact   |      256 | 0.6289     | 0.5310     |   0.6127 | 0.5522     | **0.6855**        | MiniLM Reranker |
| SciFact   |     1024 | 0.5533     | 0.5881     |   0.6432 | 0.5772     | **0.6479**        | MiniLM Reranker |

HotpotQA/QASPER: answer F1; SciFact: three-class macro-F1. Bold indicates highest point estimate, not statistical superiority.

## Efficiency

| label                                  | Dataset   |   Budget | Method             |   Quality |   Mean Evidence Tokens |   Budget Utilization |   Selected Units |   Represented Parents |   mean_redundancy |
|:---------------------------------------|:----------|---------:|:-------------------|----------:|-----------------------:|---------------------:|-----------------:|----------------------:|------------------:|
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF_V1        |  0.297642 |                 225.26 |             0.879922 |            18.16 |                  9.66 |          0.594611 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | BEA_V2             |  0.435211 |                 246.88 |             0.964375 |             7.46 |                  4.98 |          0.611181 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | FIXED_SENTENCE     |  0.366109 |                 254.84 |             0.995469 |             9.8  |                  5.28 |          0.650451 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | MINILM_CE_SENTENCE |  0.340368 |                 254.86 |             0.995547 |            10.54 |                  6.46 |          0.624712 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | QBGF_FULL          |  0.404225 |                 251.62 |             0.982891 |            11.74 |                  4.94 |          0.627003 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF_V1        |  0.344117 |                 284.64 |             0.277969 |            20.02 |                  9.72 |          0.591976 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | BEA_V2             |  0.423387 |                 960.66 |             0.938145 |            16.2  |                  9.5  |          0.572894 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | FIXED_SENTENCE     |  0.389292 |                1001.74 |             0.978262 |            36.06 |                  9.62 |          0.585496 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | MINILM_CE_SENTENCE |  0.361826 |                1000.3  |             0.976855 |            38.16 |                  9.94 |          0.574948 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | QBGF_FULL          |  0.402047 |                 995.26 |             0.971934 |            38.28 |                  9.24 |          0.573011 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | BEA_QBGF_V1        |  0.186849 |                 245.74 |             0.959922 |            20.3  |                  8.78 |          0.58434  |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | BEA_V2             |  0.261756 |                 245.04 |             0.957187 |            10.66 |                  5.28 |          0.662262 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | FIXED_SENTENCE     |  0.266794 |                 255.06 |             0.996328 |             9.34 |                  5.7  |          0.686346 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | MINILM_CE_SENTENCE |  0.264111 |                 255    |             0.996094 |            10.14 |                  6.06 |          0.653842 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | QBGF_FULL          |  0.264119 |                 252.52 |             0.986406 |            10.22 |                  5.82 |          0.68218  |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | BEA_QBGF_V1        |  0.208418 |                 472.78 |             0.461699 |            31.9  |                  9.3  |          0.587056 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | BEA_V2             |  0.290417 |                1009.1  |             0.985449 |            22.96 |                  8.14 |          0.626671 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | FIXED_SENTENCE     |  0.289603 |                1023.16 |             0.99918  |            32.2  |                  9.62 |          0.66966  |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | MINILM_CE_SENTENCE |  0.282723 |                1021.64 |             0.997695 |            38.74 |                  9.64 |          0.627879 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | QBGF_FULL          |  0.296416 |                1020.44 |             0.996523 |            24.5  |                  8.56 |          0.639274 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | BEA_QBGF_V1        |  0.612738 |                 250.08 |             0.976875 |            15.46 |                  8.16 |          0.629506 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | BEA_V2             |  0.552189 |                 242.48 |             0.947187 |             6.44 |                  4.02 |          0.741641 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | FIXED_SENTENCE     |  0.628915 |                 252.62 |             0.986797 |             6.56 |                  3.22 |          0.750613 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | MINILM_CE_SENTENCE |  0.685463 |                 252.82 |             0.987578 |             7.1  |                  3.9  |          0.726335 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | QBGF_FULL          |  0.530994 |                 250.56 |             0.97875  |             8.5  |                  3.36 |          0.721572 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | BEA_QBGF_V1        |  0.643155 |                 696.02 |             0.679707 |            33.98 |                  9.8  |          0.634118 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | BEA_V2             |  0.577204 |                 171.72 |             0.167695 |             3.88 |                  2.46 |          0.763898 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | FIXED_SENTENCE     |  0.553333 |                1020.7  |             0.996777 |            24.1  |                  8.3  |          0.718169 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | MINILM_CE_SENTENCE |  0.647868 |                1020.7  |             0.996777 |            27.76 |                  8.38 |          0.693309 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | QBGF_FULL          |  0.588105 |                1018.72 |             0.994844 |            27.7  |                  7.02 |          0.674211 |

## Descriptive point-estimate wins only

- FIXED_SENTENCE: 1 / 6
- QBGF_FULL: 1 / 6
- BEA_QBGF_V1: 0 / 6
- BEA_V2: 2 / 6
- MINILM_CE_SENTENCE: 2 / 6

Exact ties credit all tied methods, so counts can sum above six.

## Paired comparisons

| label                                  | dataset   |   budget | left               | right          |        point |      ci_low |   ci_high |     raw_p |   resamples |   seed |   n_queries |   n_clusters | metric   | test                                                                          |   holm_p |   exploratory_family_size |
|:---------------------------------------|:----------|---------:|:-------------------|:---------------|-------------:|------------:|----------:|----------:|------------:|-------:|------------:|-------------:|:---------|:------------------------------------------------------------------------------|---------:|--------------------------:|
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | MINILM_CE_SENTENCE | FIXED_SENTENCE | -0.025741    | -0.107376   | 0.0534584 | 0.558844  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | MINILM_CE_SENTENCE | QBGF_FULL      | -0.0638571   | -0.171352   | 0.039398  | 0.247275  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.0427258   | -0.0416063  | 0.133215  | 0.346065  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |      256 | MINILM_CE_SENTENCE | BEA_V2         | -0.0948431   | -0.217733   | 0.0257214 | 0.143486  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | MINILM_CE_SENTENCE | FIXED_SENTENCE | -0.0274664   | -0.0975936  | 0.0329731 | 0.476652  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | MINILM_CE_SENTENCE | QBGF_FULL      | -0.0402215   | -0.115768   | 0.0265603 | 0.289971  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.0177085   | -0.0649059  | 0.103461  | 0.689131  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | HotpotQA  |     1024 | MINILM_CE_SENTENCE | BEA_V2         | -0.0615617   | -0.147642   | 0.0146889 | 0.144186  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | MINILM_CE_SENTENCE | FIXED_SENTENCE | -0.00268351  | -0.0254521  | 0.0185513 | 0.819818  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | MINILM_CE_SENTENCE | QBGF_FULL      | -8.66263e-06 | -0.0451504  | 0.0398189 | 0.9994    |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.0772621   |  0.0136261  | 0.147747  | 0.0248975 |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 0.572643 |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |      256 | MINILM_CE_SENTENCE | BEA_V2         |  0.00235463  | -0.0220387  | 0.0257597 | 0.850015  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | MINILM_CE_SENTENCE | FIXED_SENTENCE | -0.00688011  | -0.0353136  | 0.0172327 | 0.643536  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | MINILM_CE_SENTENCE | QBGF_FULL      | -0.0136933   | -0.0393675  | 0.0106242 | 0.29757   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.0743049   |  0.00610939 | 0.148522  | 0.0467953 |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | QASPER    |     1024 | MINILM_CE_SENTENCE | BEA_V2         | -0.00769459  | -0.0518498  | 0.0326695 | 0.740026  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | MINILM_CE_SENTENCE | FIXED_SENTENCE |  0.0565479   | -0.0585276  | 0.173319  | 0.371963  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | MINILM_CE_SENTENCE | QBGF_FULL      |  0.154469    |  0.0381457  | 0.280546  | 0.0107989 |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 0.259174 |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.0727243   | -0.0731647  | 0.22464   | 0.335466  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |      256 | MINILM_CE_SENTENCE | BEA_V2         |  0.133274    | -0.00476515 | 0.2767    | 0.0953905 |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | MINILM_CE_SENTENCE | FIXED_SENTENCE |  0.0945342   |  0          | 0.197004  | 0.249775  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | MINILM_CE_SENTENCE | QBGF_FULL      |  0.059763    | -0.0384106  | 0.16502   | 0.253175  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | MINILM_CE_SENTENCE | BEA_QBGF_V1    |  0.00471261  | -0.122236   | 0.133681  | 0.943306  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |
| POST_W12_RERANKER_BASELINE_EXPLORATORY | SciFact   |     1024 | MINILM_CE_SENTENCE | BEA_V2         |  0.0706637   | -0.0844944  | 0.230917  | 0.409059  |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null | 1        |                        24 |

Existing paired cluster-bootstrap 95% intervals and cluster label-swap raw p-values; one exploratory Holm family of all 24. QASPER paper groups and SciFact cited-document components remain unchanged. These intervals omit model-training uncertainty of historical learned controls. No per-query macro-F1, pooled task score, confirmation or new method selection.

## Defensive interpretation

**Question A — MiniLM versus FIXED_SENTENCE:** higher point quality in 2/6 cells, exact point ties in 0/6. Positive nominal-CI plus Holm-p<.05 cells: none. Inspect all signs, magnitudes and uncertainty; failure to reject is not equivalence.

**Question B — MiniLM versus QBGF_FULL:** higher point quality in 2/6 cells, exact point ties in 0/6. Positive nominal-CI plus Holm-p<.05 cells: none. Inspect all signs, magnitudes and uncertainty; failure to reject is not equivalence.

**Question C — MiniLM versus BEA_QBGF_V1:** higher point quality in 6/6 cells, exact point ties in 0/6. Positive nominal-CI plus Holm-p<.05 cells: none. Inspect all signs, magnitudes and uncertainty; failure to reject is not equivalence.

**Question D — MiniLM versus BEA_V2:** higher point quality in 3/6 cells, exact point ties in 0/6. Positive nominal-CI plus Holm-p<.05 cells: none. Inspect all signs, magnitudes and uncertainty; failure to reject is not equivalence.

**Question E — “Your method is only reranking”:** this control changes the relevance scorer while holding the candidate pool, source sentences, packing budget and reader fixed. Similar scores would be compatible with relevance ranking explaining substantial performance; they would not prove equivalence. MiniLM wins would require allocation methods to justify their added complexity. MiniLM losses would show this particular static relevance scorer does not reproduce those observed gains, but would not isolate state-dependent utility as their cause. This is a pretrained English MS-MARCO model applied to three different tasks, not every possible cross-encoder. Inspect exact contexts and task-specific errors rather than selecting a favorable cell.

## Cost and diagnostic limits

CPU CE scoring is query-time computation and remains conceptual cost on cache hits. Score-cache metadata retains original measured scoring seconds and pair/truncation counts. Reader prompt/generated-token costs remain counted on reused outputs. Evidence-token savings do not imply end-to-end latency improvements. Gold retention is computed only after selection; partial/unmapped annotations are explicit. No P_help, LOO labels or downstream quality enters ranking.

| Method | Evidence value | Allocation behavior |
| --- | --- | --- |
| FIXED_SENTENCE | embedding query–sentence relevance | static global ranking + packing |
| MiniLM reranker | cross-encoder query–sentence relevance | static global ranking + packing |
| QBGF | hierarchical passage/sentence relevance mass | joint DROP/SENT/FULL budget optimization |
| BEA-V1 | relevance + coverage − redundancy | dynamic marginal selection + stopping |
| BEA-V2 | P_help × positive semantic IG | dynamic learned-utility selection + stopping |

We began with query-conditioned evidence quantity, moved to evidence identity and granularity, then tested relevance-aware and downstream-utility-aware allocation. The MiniLM cross-encoder baseline provides a conventional reranking control over the same SFR candidate pool. Together, the comparison distinguishes simple relevance reranking from budgeted granularity allocation and dynamic marginal-utility selection. The official frozen policy remains SFR top-10 → FIXED_SENTENCE → Qwen3-8B BASE_FP16 → no LoRA. This is a final POST-W12 exploratory baseline, not a replacement for the frozen method.


| Method | Evidence value | Allocation behavior |
| --- | --- | --- |
| FIXED_SENTENCE | embedding query–sentence relevance | static global ranking + packing |
| MiniLM reranker | cross-encoder query–sentence relevance | static global ranking + packing |
| QBGF | hierarchical passage/sentence relevance mass | joint DROP/SENT/FULL budget optimization |
| BEA-V1 | relevance + coverage − redundancy | dynamic marginal selection + stopping |
| BEA-V2 | P_help × positive semantic IG | dynamic learned-utility selection + stopping |

We began with query-conditioned evidence quantity, moved to evidence identity and granularity, then tested relevance-aware and downstream-utility-aware allocation. The MiniLM cross-encoder baseline provides a conventional reranking control over the same SFR candidate pool. Together, the comparison distinguishes simple relevance reranking from budgeted granularity allocation and dynamic marginal-utility selection.

The official frozen Week-12 conclusion remains **SFR top-10 → FIXED_SENTENCE → Qwen3-8B BASE_FP16 → no LoRA**. This final POST-W12 exploratory baseline does not replace that decision. Stop after the comparison; no new method, expansion or tuning is triggered.
